#Klasyfikacja URL (Phishing Detection)

## 1. Wybór i Charakterystyka Zbioru Danych

### Podstawowe Informacje
- **Nazwa zbioru danych**: Phishing Website Detection Dataset (Phishing Detection)
- **Źródło**: Kaggle - https://www.kaggle.com/datasets/shibumohapatra/book-my-show
- **Liczba rekordów**: 11 000 próbek URL
- **Liczba cech**: 30 (po usunięciu kolumny 'index')
- **Liczba klas**: 2 klasy (Legitimate: 1, Phishing: 0)
- **Typ problemu**: Klasyfikacja wieloklasowa

BookMyShow (platforma rezerwacji biletów) planuje wdrożyć system reklamowy, ale musi zapewnić bezpieczeństwo użytkowników poprzez filtrowaniem złośliwych linków. Projekt ma na celu automatyczną klasyfikację URL jako:
- **1 (Legitimate)** – bezpieczna strona
- **0 (Phishing)** – potencjalnie złośliwa strona

### Opis Cech
Zbiór zawiera 30 cech opisujących właściwości techniczne URL:
- **having_IPhaving_IP_Address**: Czy URL zawiera adres IP zamiast nazwy domeny
- **URLURL_Length**: Długość całego URL
- **Shortining_Service**: Użycie usługi skracania linków (bit.ly)
- **having_At_Symbol**: Obecność znaku '@' w URL
- **double_slash_redirecting**: Występowanie '//' w nietypowym miejscu
- **Prefix_Suffix**: Obecność '-' w nazwie domeny
- **having_Sub_Domain**: Liczba poddomen
- **SSLfinal_State**: Status certyfikatu SSL/TLS
- **Domain_registeration_length**: Długość rejestracji domeny
- **Favicon**: Czy favicon pochodzi z tej samej domeny
- **port**: Numer portu
- **HTTPS_token**: Obecność 'https' w nazwie domeny
- **Request_URL**: % zasobów z zewnętrznych domen
- **URL_of_Anchor**: % linków prowadzących do zewnętrznych domen
- **Links_in_tags**: % linków w tagach HTML
- **SFH**: Status pola formularza
- **Submitting_to_email**: Wysyłanie danych na adres e-mail
- **Abnormal_URL**: Wskaźnik niestandardowej struktury
- **Redirect**: Liczba przekierowań
- **on_mouseover**: Zdarzenia JavaScript onmouseover
- **RightClick**: Blokada kliknięcia prawym przyciskiem
- **popUpWidnow**: Wyskakujące okienka
- **Iframe**: Ramki iframe
- **age_of_domain**: Wiek domeny
- **DNSRecord**: Poprawny rekord DNS
- **web_traffic**: Ruch na stronie
- **Page_Rank**: Ranking strony
- **Google_Index**: Indeksacja przez Google
- **Links_pointing_to_page**: Liczba linków prowadzących do strony
- **Statistical_report**: Informacje z baz danych (reputacja strony)
- **Result** (TARGET): Etykieta klasyfikacji


In [23]:
# Importowanie niezbędnych bibliotek
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix, classification_report, precision_score, recall_score, f1_score, accuracy_score
from sklearn.utils import class_weight

print("Biblioteki zaimportowane")

Biblioteki zaimportowane


In [24]:
# Wgrywanie danych

df = pd.read_csv("/content/dataset.csv")

# Wyświetlenie informacji o zbiorze
print("INFORMACJE O ZBIORZE DANYCH")
print(f"Wymiary zbioru: {df.shape}")
print(f"\nPierwsze 5 wierszy:")
print(df.head())
print(f"\nTypy danych:")
print(df.dtypes)
print(f"\nBrakujące wartości:")
print(df.isna().sum())

INFORMACJE O ZBIORZE DANYCH
Wymiary zbioru: (11055, 32)

Pierwsze 5 wierszy:
  index  having_IPhaving_IP_Address  URLURL_Length  Shortining_Service  \
0     1                           0              1                   1   
1     2                           1              1                   1   
2     3                           1             -1                   1   
3     4                           1             -1                   1   
4     5                           1             -1                   0   

   having_At_Symbol  double_slash_redirecting  Prefix_Suffix  \
0                 1                         0              0   
1                 1                         1              0   
2                 1                         1              0   
3                 1                         1              0   
4                 1                         1              0   

   having_Sub_Domain  SSLfinal_State  Domain_registeration_length  ...  \
0                  

## 1.1 Przygotowanie Danych


In [25]:
# Usuwanie kolumny index, ponieważ nie niesie istotnej informacji o danych
df = df.drop(["index"], axis=1)

print(f"\nFinalnie wymiary zbioru: {df.shape}")
print(f"Kolumny: {list(df.columns)}")


Finalnie wymiary zbioru: (11055, 31)
Kolumny: ['having_IPhaving_IP_Address', 'URLURL_Length', 'Shortining_Service', 'having_At_Symbol', 'double_slash_redirecting', 'Prefix_Suffix', 'having_Sub_Domain', 'SSLfinal_State', 'Domain_registeration_length', 'Favicon', 'port', 'HTTPS_token', 'Request_URL', 'URL_of_Anchor', 'Links_in_tags', 'SFH', 'Submitting_to_email', 'Abnormal_URL', 'Redirect', 'on_mouseover', 'RightClick', 'popUpWidnow', 'Iframe', 'age_of_domain', 'DNSRecord', 'web_traffic', 'Page_Rank', 'Google_Index', 'Links_pointing_to_page', 'Statistical_report', 'Result']


## 1.2 Analiza Rozkładu Klas


In [26]:
# Analiza rozkładu klas
print("ROZKŁAD KLAS W ZBIORZE DANYCH")

class_counts = df['Result'].value_counts()
class_distribution = df['Result'].value_counts(normalize=True) * 100

print("\nLiczba próbek w każdej klasie:")
for class_label, count in class_counts.items():
    percentage = (count / len(df)) * 100
    label_name = "Legitimate" if class_label == 1 else "Phishing"
    print(f"  {label_name} ({class_label}): {count} próbek ({percentage:.2f}%)")

# Sprawdzenie niezrównoważenia klas
min_class = class_counts.min()
max_class = class_counts.max()
imbalance_ratio = max_class / min_class
print(f"\nStosunek niezrównoważenia: {imbalance_ratio:.2f}:1")

if imbalance_ratio > 1.5:
    print(f"Klasy są niezrównoważone (stosunek > 1.5:1)")
    print("Będą zastosowane techniki radzenia sobie z niezrównoważeniem")
else:
    print(f"Klasy są w miarę zrównoważone (stosunek ≤ 1.5:1)")

ROZKŁAD KLAS W ZBIORZE DANYCH

Liczba próbek w każdej klasie:
  Legitimate (1): 6157 próbek (55.69%)
  Phishing (0): 4898 próbek (44.31%)

Stosunek niezrównoważenia: 1.26:1
Klasy są w miarę zrównoważone (stosunek ≤ 1.5:1)


## 2. Wybór Metryki Ewaluacyjnej - Precision vs Recall

### Analiza Kosztów Błędów

W kontekście problemu klasyfikacji URL (Phishing Detection) dla platformy BookMyShow, należy rozważyć koszty dwóch typów błędów:

**False Positive (FP)** - Bezpieczny URL zaklasyfikowany jako phishing/podejrzany:
- Użytkownik nie widzi legalnej reklamy
- Strata dochodów dla platformy (brak wyświetlenia reklamy)
- Złe doświadczenie użytkownika (zablokowanie legalnego linku)
- Koszt: ŚREDNI

**False Negative (FN)** - Phishing URL zaklasyfikowany jako bezpieczny:
- Użytkownik klika na złośliwy link
- Potencjalna kradzież danych użytkownika
- Szkoda dla reputacji platformy
- Możliwe problemy prawne i bezpieczeństwa
- Koszt: BARDZO WYSOKI

### Uzasadnienie Wyboru Metryki

**Recall (czułość) zostaje wybrany jako metryka priorytetowa**

Recall = TP / (TP + FN) - mierzy jaki procent rzeczywistych phishingowych linków został wykryty

**Uzasadnienie:**
1. Koszt False Negative (niewychwycony phishing) jest znacznie wyższy niż koszt False Positive
2. Bezpieczeństwo użytkowników jest priorytetem
3. Lepiej zablokować kilka legalnych reklam (FP) niż przeopuścić phishing (FN)
4. W системах bezpieczeństwa Recall jest zazwyczaj ważniejszy niż Precision

**Jednak będziemy śledzić wszystkie metryki** (Precision, Recall, F1-score) aby znaleźć model z optymalnym balansem.


## 3. Podział Zbioru Danych


In [27]:
# Separacja cech (X) i etykiet (y)
X = df.drop(['Result'], axis=1)
y = df['Result']

print("PODZIAŁ ZBIORU DANYCH")
print(f"\nCechy (X): {X.shape}")
print(f"Etykiety (y): {y.shape}")

# Podział na zbiory treningowy i testowy
# Stratified = zachowanie proporcji klas w obu zbiorach
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,  # 30% na testy
    random_state=42,  # Dla powtarzalności
    stratify=y
)

print(f"\nZbiór treningowy: {X_train.shape[0]} próbek (70%)")
print(f"Zbiór testowy: {X_test.shape[0]} próbek (30%)")

# Sprawdzenie rozkładu klas w zbiorach
print(f"\nRozkład klas w zbiorze treningowym:")
for class_label in sorted(y_train.unique()):
    count = (y_train == class_label).sum()
    percentage = (count / len(y_train)) * 100
    label_name = "Legitimate" if class_label == 1 else "Phishing"
    print(f"  {label_name} ({class_label}): {count} próbek ({percentage:.2f}%)")

print(f"\nRozkład klas w zbiorze testowym:")
for class_label in sorted(y_test.unique()):
    count = (y_test == class_label).sum()
    percentage = (count / len(y_test)) * 100
    label_name = "Legitimate" if class_label == 1 else "Phishing"
    print(f"  {label_name} ({class_label}): {count} próbek ({percentage:.2f}%)")

PODZIAŁ ZBIORU DANYCH

Cechy (X): (11055, 30)
Etykiety (y): (11055,)

Zbiór treningowy: 7738 próbek (70%)
Zbiór testowy: 3317 próbek (30%)

Rozkład klas w zbiorze treningowym:
  Phishing (0): 3428 próbek (44.30%)
  Legitimate (1): 4310 próbek (55.70%)

Rozkład klas w zbiorze testowym:
  Phishing (0): 1470 próbek (44.32%)
  Legitimate (1): 1847 próbek (55.68%)


## 4. Testowanie Algorytmów

Będą testowane następujące algorytmy:
1. Regresja Logistyczna
2. k-Nearest Neighbors (k-NN)
3. Drzewo Decyzyjne
4. Random Forest
5. Naive Bayes
6. Gradient Boosting

Dla każdego algorytmu będzie przeprowadzona:
- Walidacja krzyżowa (5-fold)
- Strojenie hiperparametrów (w wybranych przypadkach)
- Obliczenie metryk (Precision, Recall, F1-score, Accuracy)


In [29]:
# Definicja algorytmów z hiperparametrami
algorithms = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=42
    ),
    'k-NN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=15,
        min_samples_split=10,
        class_weight='balanced',
        random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        min_samples_split=5,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ),
    'Naive Bayes': GaussianNB(),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        random_state=42
    )
}

print(f"Liczba algorytmów do testowania: {len(algorithms)}")
print(f"Algorytmy: {list(algorithms.keys())}")

Liczba algorytmów do testowania: 6
Algorytmy: ['Logistic Regression', 'k-NN (k=5)', 'Decision Tree', 'Random Forest', 'Naive Bayes', 'Gradient Boosting']


In [30]:
# Trenowanie i ewaluacja wszystkich algorytmów
results = []
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("TRENING I EWALUACJA ALGORYTMÓW")


for algo_name, model in algorithms.items():
    print(f"\nTrenowanie: {algo_name}...")

    # Walidacja krzyżowa
    cv_scores_accuracy = cross_val_score(
        model, X_train, y_train,
        cv=kfold,
        scoring='accuracy',
        n_jobs=-1
    )

    cv_scores_recall = cross_val_score(
        model, X_train, y_train,
        cv=kfold,
        scoring='recall_weighted',
        n_jobs=-1
    )

    cv_scores_precision = cross_val_score(
        model, X_train, y_train,
        cv=kfold,
        scoring='precision_weighted',
        n_jobs=-1
    )

    # Trenowanie na pełnym zbiorze treningowym
    model.fit(X_train, y_train)

    # Predykcja na zbiorze testowym
    y_pred = model.predict(X_test)

    # Obliczenie metryk
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    # Zapis wyników
    results.append({
        'Algorithm': algo_name,
        'CV_Accuracy': cv_scores_accuracy.mean(),
        'CV_Recall': cv_scores_recall.mean(),
        'CV_Precision': cv_scores_precision.mean(),
        'Test_Accuracy': accuracy,
        'Test_Precision': precision,
        'Test_Recall': recall,
        'Test_F1': f1,
        'Model': model
    })

    print(f" CV Accuracy: {cv_scores_accuracy.mean():.4f} (+/- {cv_scores_accuracy.std():.4f})")
    print(f" Test Accuracy: {accuracy:.4f}")
    print(f" Test Recall: {recall:.4f}")
    print(f" Test Precision: {precision:.4f}")
    print(f" Test F1-score: {f1:.4f}")


print("Trening 6 modeli zakończony")


TRENING I EWALUACJA ALGORYTMÓW

Trenowanie: Logistic Regression...
 CV Accuracy: 0.9112 (+/- 0.0095)
 Test Accuracy: 0.9105
 Test Recall: 0.9105
 Test Precision: 0.9105
 Test F1-score: 0.9105

Trenowanie: k-NN (k=5)...
 CV Accuracy: 0.9457 (+/- 0.0043)
 Test Accuracy: 0.9494
 Test Recall: 0.9494
 Test Precision: 0.9494
 Test F1-score: 0.9493

Trenowanie: Decision Tree...
 CV Accuracy: 0.9446 (+/- 0.0031)
 Test Accuracy: 0.9536
 Test Recall: 0.9536
 Test Precision: 0.9537
 Test F1-score: 0.9536

Trenowanie: Random Forest...
 CV Accuracy: 0.9589 (+/- 0.0033)
 Test Accuracy: 0.9671
 Test Recall: 0.9671
 Test Precision: 0.9671
 Test F1-score: 0.9671

Trenowanie: Naive Bayes...
 CV Accuracy: 0.5837 (+/- 0.0027)
 Test Accuracy: 0.5855
 Test Recall: 0.5855
 Test Precision: 0.7858
 Test F1-score: 0.5286

Trenowanie: Gradient Boosting...
 CV Accuracy: 0.9601 (+/- 0.0062)
 Test Accuracy: 0.9626
 Test Recall: 0.9626
 Test Precision: 0.9627
 Test F1-score: 0.9626
Trening 6 modeli zakończony


## 4.1 Porównanie Wyników - Tabela Podsumowania


In [31]:
# Tworzenie dataframe z wynikami
results_df = pd.DataFrame([
    {
        'Algorithm': r['Algorithm'],
        'CV_Acc': f"{r['CV_Accuracy']:.4f}",
        'CV_Recall': f"{r['CV_Recall']:.4f}",
        'Test_Acc': f"{r['Test_Accuracy']:.4f}",
        'Test_Recall': f"{r['Test_Recall']:.4f}",
        'Test_Prec': f"{r['Test_Precision']:.4f}",
        'Test_F1': f"{r['Test_F1']:.4f}"
    }
    for r in results
])

print("TABELA PORÓWNAWCZA ALGORYTMÓW")

print(results_df.to_string(index=False))

# Znalezienie najlepszego modelu (po Recall)
best_model_idx = max(range(len(results)), key=lambda i: results[i]['Test_Recall'])
best_model_name = results[best_model_idx]['Algorithm']
best_model = results[best_model_idx]['Model']

print(f"\nNajlepszy model (po Recall): {best_model_name}")
print(f"  Recall: {results[best_model_idx]['Test_Recall']:.4f}")
print(f"  Precision: {results[best_model_idx]['Test_Precision']:.4f}")
print(f"  F1-score: {results[best_model_idx]['Test_F1']:.4f}")

TABELA PORÓWNAWCZA ALGORYTMÓW
          Algorithm CV_Acc CV_Recall Test_Acc Test_Recall Test_Prec Test_F1
Logistic Regression 0.9112    0.9112   0.9105      0.9105    0.9105  0.9105
         k-NN (k=5) 0.9457    0.9457   0.9494      0.9494    0.9494  0.9493
      Decision Tree 0.9446    0.9446   0.9536      0.9536    0.9537  0.9536
      Random Forest 0.9589    0.9589   0.9671      0.9671    0.9671  0.9671
        Naive Bayes 0.5837    0.5837   0.5855      0.5855    0.7858  0.5286
  Gradient Boosting 0.9601    0.9601   0.9626      0.9626    0.9627  0.9626

Najlepszy model (po Recall): Random Forest
  Recall: 0.9671
  Precision: 0.9671
  F1-score: 0.9671


## 5. Predykcja i Ewaluacja - Najlepszy Model


In [32]:
# Użycie najlepszego modelu

print(f"EWALUACJA NAJLEPSZEGO MODELU: {best_model_name}")

# Predykcja
y_pred_best = best_model.predict(X_test)

# Metryki
accuracy = accuracy_score(y_test, y_pred_best)
precision = precision_score(y_test, y_pred_best, average='weighted', zero_division=0)
recall = recall_score(y_test, y_pred_best, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred_best, average='weighted', zero_division=0)

print(f"\nMetryki na zbiorze testowym:")
print(f"  Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"  Precision: {precision:.4f} ({precision*100:.2f}%)")
print(f"  Recall: {recall:.4f} ({recall*100:.2f}%) [PRIORYTET]")
print(f"  F1-score: {f1:.4f}")

# Macierz pomyłek
print(f"\nMacierz pomyłek (Confusion Matrix):")
cm = confusion_matrix(y_test, y_pred_best)
print(cm)
print(f"\nInterpretacja:")
print(f"  True Negatives (TN): {cm[0, 0]}")
print(f"  False Positives (FP): {cm[0, 1]}")
print(f"  False Negatives (FN): {cm[1, 0]}")
print(f"  True Positives (TP): {cm[1, 1]}")

EWALUACJA NAJLEPSZEGO MODELU: Random Forest

Metryki na zbiorze testowym:
  Accuracy: 0.9671 (96.71%)
  Precision: 0.9671 (96.71%)
  Recall: 0.9671 (96.71%) [PRIORYTET]
  F1-score: 0.9671

Macierz pomyłek (Confusion Matrix):
[[1417   53]
 [  56 1791]]

Interpretacja:
  True Negatives (TN): 1417
  False Positives (FP): 53
  False Negatives (FN): 56
  True Positives (TP): 1791


## 5.1 Analiza Wyników w Kontekście Celów Projektu


In [33]:
print("ANALIZA WYNIKÓW")

# Analiza relative to chosen metric
print(f"\n1. RECALL (metryka priorytetowa):")
print(f"   Wartość: {recall:.4f} ({recall*100:.2f}%)")
print(f"   Interpretacja: Model poprawnie wykrywa {recall*100:.2f}% rzeczywistych zagrożeń (phishingu)")

if recall > 0.90:
    print(f"   Doskonały wynik - model bardzo dobrze chroni użytkowników")
elif recall > 0.85:
    print(f"   Bardzo dobry wynik - model adekwatnie chroni użytkowników")
elif recall > 0.80:
    print(f"   Dobry wynik - może wymagać poprawy w ochronie")
else:
    print(f"   Niewystarczający wynik - wymaga znacznej poprawy")

print(f"\n2. PRECISION (minimalizacja fałszywych alarmów):")
print(f"   Wartość: {precision:.4f} ({precision*100:.2f}%)")
print(f"   Interpretacja: {precision*100:.2f}% linków zaklasyfikowanych jako bezpieczne rzeczywiście są bezpieczne")

if precision > 0.90:
    print(f"    Doskonały wynik - bardzo mało fałszywych alarmów")
elif precision > 0.85:
    print(f"    Bardzo dobry wynik - akceptowalna liczba fałszywych alarmów")
else:
    print(f"    Wiele fałszywych alarmów - może frustrować użytkowników")

print(f"\n3. F1-SCORE (balans między Recall i Precision):")
print(f"   Wartość: {f1:.4f} ({f1*100:.2f}%)")
print(f"   Interpretacja: Średnia harmoniczna Precision i Recall")

print(f"\n4. ACCURACY (ogólna dokładność):")
print(f"   Wartość: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"   Interpretacja: {accuracy*100:.2f}% wszystkich klasyfikacji było poprawnych")

# Analiza błędów
print(f"\n5. ANALIZA BŁĘDÓW:")
fp_rate = cm[0, 1] / (cm[0, 0] + cm[0, 1])
fn_rate = cm[1, 0] / (cm[1, 0] + cm[1, 1])
print(f"   False Positive Rate: {fp_rate:.4f} ({fp_rate*100:.2f}%)")
print(f"   (% bezpiecznych linków zaklasyfikowanych jako zagrożenie)")
print(f"\n   False Negative Rate: {fn_rate:.4f} ({fn_rate*100:.2f}%)")
print(f"   (% zagrożeń zaklasyfikowanych jako bezpieczne)")

if fn_rate < fp_rate:
    print(f"   Model lepiej minimalizuje brakujące zagrożenia (False Negatives)")
    print(f"     To jest pożądane w systemach bezpieczeństwa!")
else:
    print(f"   Model generuje więcej False Negatives niż False Positives")
    print(f"     To stanowi zagrożenie dla bezpieczeństwa")

ANALIZA WYNIKÓW

1. RECALL (metryka priorytetowa):
   Wartość: 0.9671 (96.71%)
   Interpretacja: Model poprawnie wykrywa 96.71% rzeczywistych zagrożeń (phishingu)
   Doskonały wynik - model bardzo dobrze chroni użytkowników

2. PRECISION (minimalizacja fałszywych alarmów):
   Wartość: 0.9671 (96.71%)
   Interpretacja: 96.71% linków zaklasyfikowanych jako bezpieczne rzeczywiście są bezpieczne
    Doskonały wynik - bardzo mało fałszywych alarmów

3. F1-SCORE (balans między Recall i Precision):
   Wartość: 0.9671 (96.71%)
   Interpretacja: Średnia harmoniczna Precision i Recall

4. ACCURACY (ogólna dokładność):
   Wartość: 0.9671 (96.71%)
   Interpretacja: 96.71% wszystkich klasyfikacji było poprawnych

5. ANALIZA BŁĘDÓW:
   False Positive Rate: 0.0361 (3.61%)
   (% bezpiecznych linków zaklasyfikowanych jako zagrożenie)

   False Negative Rate: 0.0303 (3.03%)
   (% zagrożeń zaklasyfikowanych jako bezpieczne)
   Model lepiej minimalizuje brakujące zagrożenia (False Negatives)
     To jest 

## 6. Wyniki i Wnioski


1. CHARAKTERYSTYKA ZBIORU DANYCH

Liczba próbek: 11 000 URL

Liczba cech: 30

Liczba klas: 2 (Legitimate, Phishing)

Rozkład klas: Zrównoważony

Brakujące wartości: Brak

2. WYBÓR METRYKI EWALUACYJNEJ

Metryka priorytetowa: RECALL

Uzasadnienie: False Negative (niewychwycony phishing) jest znacznie bardziej kosztowny niż False Positive

Cel: Maksymalnie bezpieczeństwo użytkowników

3. PODZIAŁ ZBIORU DANYCH

Zbiór treningowy: 70% (7 700 próbek) - stratyfikowany

Zbiór testowy: 30% (3 300 próbek) - stratyfikowany

Wykorzystano: StandardScaler, class_weight='balanced'

4. PRZETESTOWANE ALGORYTMY

In [34]:
for i, r in enumerate(results, 1):
    print(f"   {i}. {r['Algorithm']:25} - Recall: {r['Test_Recall']:.4f}, F1: {r['Test_F1']:.4f}")

print(f"\n5. NAJLEPSZY MODEL:")
print(f"   - Algorytm: {best_model_name}")
print(f"   - Accuracy: {results[best_model_idx]['Test_Accuracy']:.4f}")
print(f"   - Precision: {results[best_model_idx]['Test_Precision']:.4f}")
print(f"   - Recall: {results[best_model_idx]['Test_Recall']:.4f}")
print(f"   - F1-score: {results[best_model_idx]['Test_F1']:.4f}")

   1. Logistic Regression       - Recall: 0.9105, F1: 0.9105
   2. k-NN (k=5)                - Recall: 0.9494, F1: 0.9493
   3. Decision Tree             - Recall: 0.9536, F1: 0.9536
   4. Random Forest             - Recall: 0.9671, F1: 0.9671
   5. Naive Bayes               - Recall: 0.5855, F1: 0.5286
   6. Gradient Boosting         - Recall: 0.9626, F1: 0.9626

5. NAJLEPSZY MODEL:
   - Algorytm: Random Forest
   - Accuracy: 0.9671
   - Precision: 0.9671
   - Recall: 0.9671
   - F1-score: 0.9671


6. CO DZIAŁAŁO NAJLEPIEJ

* Modelowanie ensemble (Random Forest, Gradient Boosting) pokazało lepsze wyniki niż modele liniowe

* Standaryzacja cech (StandardScaler) była istotna dla wydajności

* Stratifikowany podział zbioru zachował rozkład klas

* Walidacja krzyżowa (5-fold) potwierdziła stabilność modelu

7. JAKIE BYŁY PROBLEMY

* Niezrównoważenie klas nie było znaczącym problemem w tym zbiorze

* Wszystkie algorytmy osiągnęły zadowalające wyniki

* Różnice w wydajności między top modelami są nieznaczne

8. JAK MOŻNA POPRAWIĆ WYNIKI

a) Feature Engineering

* Analiza korelacji cech

* Selekcja najistotniejszych cech (feature importance)

* Tworzenie nowych cech (feature creation)


b) Równowaga między Recall a Precision

* Zmiana threshold dla klasyfikacji

* Wdrożenie cost-sensitive learning

c) Zbieranie Danych

* Więcej próbek phishingowych URLs

* Aktualizacja danych (phishing evolves)

* Real-world validation

d) Ensemble Methods

* Stacking różnych modeli

* Voting classifier

* Blending

9. REKOMENDACJE

In [35]:
print(f"   Model {best_model_name} jest gotowy do produkcji")
print(f"   Recall {results[best_model_idx]['Test_Recall']:.1%} zapewnia odpowiednią ochronę")
print(f"   Precision {results[best_model_idx]['Test_Precision']:.1%} minimalizuje rozczarowanie")
print(f"   Okresowe przetrening modelu z nowymi danymi phishingowymi")
print(f"   Monitoring wydajności w produkcji")

   Model Random Forest jest gotowy do produkcji
   Recall 96.7% zapewnia odpowiednią ochronę
   Precision 96.7% minimalizuje rozczarowanie
   Okresowe przetrening modelu z nowymi danymi phishingowymi
   Monitoring wydajności w produkcji
